# GAN_PlankEye v2 — Kaggle / PyTorch / 512×512

Ce notebook remplace la version Paddle/AI Studio par une version PyTorch CUDA adaptée à Kaggle.

- U-Net conditionnel
- PatchGAN
- bruit latent
- `lambda_L1 = 50`
- labels polygonaux `classe x1 y1 ... x4 y4`
- checkpoints `best.pt` et `last.pt`
- génération multi-planches 1 à 7

In [2]:
print ("ok")

ok


In [3]:
import torch, os
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("GPU dispo:", torch.cuda.is_available())
print("Nb GPU   :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

if not torch.cuda.is_available():
    raise RuntimeError("Active un GPU dans Kaggle > Settings > Accelerator.")

PyTorch : 2.10.0+cu128
CUDA    : 12.8
GPU dispo: True
Nb GPU   : 2
0 Tesla T4
1 Tesla T4


In [5]:
from pathlib import Path

PROJECT = Path("/kaggle/working/GAN_PlankEye_v2")
PROJECT.mkdir(parents=True, exist_ok=True)

FILES = {'config.py': 'from pathlib import Path\n\nPROJECT_DIR = Path("/kaggle/working/GAN_PlankEye_v2")\n\n# Entraînement\nIMAGE_SIZE = 512\nSEED = 42\nLATENT_CHANNELS = 3\nBASE_CHANNELS = 64\nBATCH_PER_GPU = 4\nNUM_WORKERS = 2\nEPOCHS = 150\nLR_G = 2e-4\nLR_D = 2e-4\nBETA1 = 0.5\nBETA2 = 0.999\nLAMBDA_L1 = 50.0\nVAL_RATIO = 0.10\n\n# Dossiers\nPAIRED_DIR = PROJECT_DIR / "data" / "paired"\nRUN_DIR = PROJECT_DIR / "runs" / "plankgan_multi_512"\nCHECKPOINT_DIR = RUN_DIR / "checkpoints"\nSAMPLE_DIR = RUN_DIR / "samples"\nGENERATED_DIR = PROJECT_DIR / "generated"\n', 'utils.py': 'from __future__ import annotations\n\nimport random\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\n\nIMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\n\n\ndef seed_everything(seed: int = 42) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef natural_key(path: Path):\n    import re\n    return [\n        int(part) if part.isdigit() else part.lower()\n        for part in re.split(r"(\\d+)", path.stem)\n    ]\n\n\ndef read_polygon_label(label_path: Path):\n    """\n    Format attendu par ligne:\n        classe x1 y1 x2 y2 x3 y3 x4 y4\n\n    Coordonnées normalisées dans [0, 1].\n    """\n    polygons = []\n\n    if not label_path.exists():\n        return polygons\n\n    for line in label_path.read_text(encoding="utf-8").splitlines():\n        line = line.strip()\n        if not line:\n            continue\n\n        parts = line.split()\n        if len(parts) < 9:\n            continue\n\n        cls = int(float(parts[0]))\n        coords = list(map(float, parts[1:9]))\n\n        pts = np.asarray(coords, dtype=np.float32).reshape(4, 2)\n        pts = np.clip(pts, 0.0, 1.0)\n\n        polygons.append((cls, pts))\n\n    return polygons\n\n\ndef polygons_to_mask(polygons, width: int, height: int) -> np.ndarray:\n    mask = np.zeros((height, width), dtype=np.uint8)\n\n    for _, pts_norm in polygons:\n        pts = pts_norm.copy()\n        pts[:, 0] *= max(width - 1, 1)\n        pts[:, 1] *= max(height - 1, 1)\n        pts = np.round(pts).astype(np.int32)\n        cv2.fillPoly(mask, [pts], 255)\n\n    return mask\n\n\ndef denorm_image(t: torch.Tensor) -> torch.Tensor:\n    """[-1, 1] -> [0, 1]."""\n    return (t.clamp(-1, 1) + 1.0) * 0.5\n\n\ndef save_triplet_grid(mask, fake, real, path: Path, max_items: int = 4):\n    """\n    Sauvegarde une grille:\n      ligne 1 : masques\n      ligne 2 : images GAN\n      ligne 3 : images réelles\n    """\n    from torchvision.utils import make_grid, save_image\n\n    n = min(max_items, mask.shape[0])\n\n    mask_rgb = mask[:n].repeat(1, 3, 1, 1)\n    fake = denorm_image(fake[:n])\n    real = denorm_image(real[:n])\n\n    grid = torch.cat([mask_rgb, fake, real], dim=0)\n    grid = make_grid(grid, nrow=n, padding=2)\n\n    path.parent.mkdir(parents=True, exist_ok=True)\n    save_image(grid, str(path))\n', 'dataset.py': 'from __future__ import annotations\n\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom torchvision.transforms import functional as TF\n\n\nclass PairedPlankDataset(Dataset):\n    def __init__(self, root: str | Path, augment: bool = False):\n        self.root = Path(root)\n        self.image_dir = self.root / "images"\n        self.mask_dir = self.root / "masks"\n\n        if not self.image_dir.exists():\n            raise FileNotFoundError(f"Dossier images introuvable: {self.image_dir}")\n\n        self.images = sorted(\n            p for p in self.image_dir.iterdir()\n            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\n        )\n\n        valid = []\n        for img in self.images:\n            mask = self.mask_dir / f"{img.stem}.png"\n            if mask.exists():\n                valid.append(img)\n\n        self.images = valid\n        self.augment = augment\n\n        if not self.images:\n            raise RuntimeError(f"Aucune paire image/masque trouvée dans {root}")\n\n    def __len__(self):\n        return len(self.images)\n\n    def __getitem__(self, index):\n        image_path = self.images[index]\n        mask_path = self.mask_dir / f"{image_path.stem}.png"\n\n        image = Image.open(image_path).convert("RGB")\n        mask = Image.open(mask_path).convert("L")\n\n        # Augmentations géométriques synchronisées.\n        if self.augment:\n            if random.random() < 0.5:\n                image = TF.hflip(image)\n                mask = TF.hflip(mask)\n\n            if random.random() < 0.25:\n                image = TF.vflip(image)\n                mask = TF.vflip(mask)\n\n        image = TF.to_tensor(image)\n        image = image * 2.0 - 1.0\n\n        mask = TF.to_tensor(mask)\n        mask = (mask > 0.5).float()\n\n        return {\n            "image": image,\n            "mask": mask,\n            "name": image_path.stem,\n        }\n', 'models.py': 'from __future__ import annotations\n\nimport torch\nimport torch.nn as nn\n\n\ndef _down(in_ch, out_ch, normalize=True):\n    layers = [\n        nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=not normalize)\n    ]\n    if normalize:\n        layers.append(nn.InstanceNorm2d(out_ch, affine=True))\n    layers.append(nn.LeakyReLU(0.2, inplace=True))\n    return nn.Sequential(*layers)\n\n\ndef _up(in_ch, out_ch, dropout=0.0):\n    layers = [\n        nn.ConvTranspose2d(\n            in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=False\n        ),\n        nn.InstanceNorm2d(out_ch, affine=True),\n        nn.ReLU(inplace=True),\n    ]\n    if dropout > 0:\n        layers.append(nn.Dropout(dropout))\n    return nn.Sequential(*layers)\n\n\nclass UNetGenerator(nn.Module):\n    """\n    Générateur conditionnel.\n    Entrée = masque (1 canal) + bruit latent spatial.\n    Sortie = RGB dans [-1, 1].\n\n    Architecture prévue pour 512x512.\n    """\n\n    def __init__(self, latent_channels=3, base=64):\n        super().__init__()\n        in_ch = 1 + latent_channels\n\n        self.d1 = _down(in_ch, base, normalize=False)       # 512 -> 256\n        self.d2 = _down(base, base * 2)                    # 256 -> 128\n        self.d3 = _down(base * 2, base * 4)                # 128 -> 64\n        self.d4 = _down(base * 4, base * 8)                # 64 -> 32\n        self.d5 = _down(base * 8, base * 8)                # 32 -> 16\n        self.d6 = _down(base * 8, base * 8)                # 16 -> 8\n        self.d7 = _down(base * 8, base * 8, normalize=False)  # 8 -> 4\n\n        self.u1 = _up(base * 8, base * 8, dropout=0.5)     # 4 -> 8\n        self.u2 = _up(base * 16, base * 8, dropout=0.5)    # 8 -> 16\n        self.u3 = _up(base * 16, base * 8, dropout=0.5)    # 16 -> 32\n        self.u4 = _up(base * 16, base * 4)                 # 32 -> 64\n        self.u5 = _up(base * 8, base * 2)                  # 64 -> 128\n        self.u6 = _up(base * 4, base)                      # 128 -> 256\n\n        self.out = nn.Sequential(\n            nn.ConvTranspose2d(\n                base * 2, 3, kernel_size=4, stride=2, padding=1\n            ),\n            nn.Tanh(),\n        )\n\n    def forward(self, mask, noise):\n        x = torch.cat([mask, noise], dim=1)\n\n        d1 = self.d1(x)\n        d2 = self.d2(d1)\n        d3 = self.d3(d2)\n        d4 = self.d4(d3)\n        d5 = self.d5(d4)\n        d6 = self.d6(d5)\n        d7 = self.d7(d6)\n\n        u1 = self.u1(d7)\n        u1 = torch.cat([u1, d6], dim=1)\n\n        u2 = self.u2(u1)\n        u2 = torch.cat([u2, d5], dim=1)\n\n        u3 = self.u3(u2)\n        u3 = torch.cat([u3, d4], dim=1)\n\n        u4 = self.u4(u3)\n        u4 = torch.cat([u4, d3], dim=1)\n\n        u5 = self.u5(u4)\n        u5 = torch.cat([u5, d2], dim=1)\n\n        u6 = self.u6(u5)\n        u6 = torch.cat([u6, d1], dim=1)\n\n        return self.out(u6)\n\n\nclass PatchDiscriminator(nn.Module):\n    """\n    PatchGAN conditionnel:\n        entrée = masque + image RGB.\n    """\n\n    def __init__(self, base=64):\n        super().__init__()\n\n        self.net = nn.Sequential(\n            nn.Conv2d(4, base, 4, 2, 1),\n            nn.LeakyReLU(0.2, inplace=True),\n\n            nn.Conv2d(base, base * 2, 4, 2, 1, bias=False),\n            nn.InstanceNorm2d(base * 2, affine=True),\n            nn.LeakyReLU(0.2, inplace=True),\n\n            nn.Conv2d(base * 2, base * 4, 4, 2, 1, bias=False),\n            nn.InstanceNorm2d(base * 4, affine=True),\n            nn.LeakyReLU(0.2, inplace=True),\n\n            nn.Conv2d(base * 4, base * 8, 4, 1, 1, bias=False),\n            nn.InstanceNorm2d(base * 8, affine=True),\n            nn.LeakyReLU(0.2, inplace=True),\n\n            nn.Conv2d(base * 8, 1, 4, 1, 1),\n        )\n\n    def forward(self, mask, image):\n        return self.net(torch.cat([mask, image], dim=1))\n\n\ndef init_weights(module):\n    classname = module.__class__.__name__\n\n    if "Conv" in classname and hasattr(module, "weight") and module.weight is not None:\n        nn.init.normal_(module.weight.data, 0.0, 0.02)\n\n    if "Norm" in classname and hasattr(module, "weight") and module.weight is not None:\n        nn.init.normal_(module.weight.data, 1.0, 0.02)\n\n    if hasattr(module, "bias") and module.bias is not None:\n        nn.init.constant_(module.bias.data, 0.0)\n', 'prepare_dataset.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nfrom collections import Counter\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nfrom PIL import Image\n\nfrom utils import IMAGE_EXTENSIONS, natural_key, polygons_to_mask, read_polygon_label\n\n\ndef find_dataset_candidates(root: Path):\n    """\n    Recherche automatiquement les dossiers:\n        .../images\n        .../labels\n    dans /kaggle/input.\n    """\n    candidates = []\n\n    for image_dir in root.rglob("images"):\n        if not image_dir.is_dir():\n            continue\n\n        label_dir = image_dir.parent / "labels"\n        if not label_dir.is_dir():\n            continue\n\n        images = [\n            p for p in image_dir.iterdir()\n            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS\n        ]\n\n        paired = sum(\n            1 for p in images\n            if (label_dir / f"{p.stem}.txt").exists()\n        )\n\n        if paired:\n            candidates.append((paired, image_dir.parent))\n\n    candidates.sort(reverse=True, key=lambda x: x[0])\n    return candidates\n\n\ndef prepare(source: Path, out: Path, size: int, overwrite: bool):\n    image_dir = source / "images"\n    label_dir = source / "labels"\n\n    if not image_dir.exists() or not label_dir.exists():\n        raise FileNotFoundError(\n            "Le dossier source doit contenir `images/` et `labels/`.\\n"\n            f"Source reçue: {source}"\n        )\n\n    if overwrite and out.exists():\n        shutil.rmtree(out)\n\n    out_images = out / "images"\n    out_masks = out / "masks"\n    out_labels = out / "labels"\n\n    out_images.mkdir(parents=True, exist_ok=True)\n    out_masks.mkdir(parents=True, exist_ok=True)\n    out_labels.mkdir(parents=True, exist_ok=True)\n\n    image_paths = sorted(\n        [\n            p for p in image_dir.iterdir()\n            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS\n        ],\n        key=natural_key,\n    )\n\n    kept = 0\n    missing_label = 0\n    empty_label = 0\n    count_distribution = Counter()\n\n    metadata = []\n\n    for image_path in image_paths:\n        label_path = label_dir / f"{image_path.stem}.txt"\n\n        if not label_path.exists():\n            missing_label += 1\n            continue\n\n        polygons = read_polygon_label(label_path)\n\n        if not polygons:\n            empty_label += 1\n            continue\n\n        image = Image.open(image_path).convert("RGB")\n        image = image.resize((size, size), Image.Resampling.LANCZOS)\n\n        mask = polygons_to_mask(polygons, size, size)\n\n        out_image = out_images / f"{image_path.stem}.jpg"\n        out_mask = out_masks / f"{image_path.stem}.png"\n        out_label = out_labels / f"{image_path.stem}.txt"\n\n        image.save(out_image, quality=95)\n        cv2.imwrite(str(out_mask), mask)\n\n        # Les coordonnées sont normalisées: le resize ne les change pas.\n        shutil.copy2(label_path, out_label)\n\n        count_distribution[len(polygons)] += 1\n\n        metadata.append(\n            {\n                "stem": image_path.stem,\n                "source_image": str(image_path),\n                "source_label": str(label_path),\n                "plank_count": len(polygons),\n            }\n        )\n\n        kept += 1\n\n    info = {\n        "source": str(source),\n        "output": str(out),\n        "size": size,\n        "pairs": kept,\n        "missing_label": missing_label,\n        "empty_label": empty_label,\n        "plank_count_distribution": dict(sorted(count_distribution.items())),\n        "items": metadata,\n    }\n\n    (out / "metadata.json").write_text(\n        json.dumps(info, indent=2, ensure_ascii=False),\n        encoding="utf-8",\n    )\n\n    print("=" * 72)\n    print("PRÉPARATION TERMINÉE")\n    print("=" * 72)\n    print(f"Source                : {source}")\n    print(f"Sortie                : {out}")\n    print(f"Taille                : {size}x{size}")\n    print(f"Paires conservées     : {kept}")\n    print(f"Sans label            : {missing_label}")\n    print(f"Labels vides/invalides: {empty_label}")\n    print("Distribution nb planches:")\n    for n, c in sorted(count_distribution.items()):\n        print(f"  {n}: {c}")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--source", type=str, default="")\n    parser.add_argument("--search-root", type=str, default="/kaggle/input")\n    parser.add_argument("--out", type=str, required=True)\n    parser.add_argument("--size", type=int, default=512)\n    parser.add_argument("--overwrite", action="store_true")\n    args = parser.parse_args()\n\n    if args.source:\n        source = Path(args.source)\n    else:\n        candidates = find_dataset_candidates(Path(args.search_root))\n\n        if not candidates:\n            raise RuntimeError(\n                "Aucun dataset `images/ + labels/` trouvé dans /kaggle/input.\\n"\n                "Passe explicitement --source /kaggle/input/.../ton_dataset"\n            )\n\n        print("Datasets candidats:")\n        for paired, path in candidates[:10]:\n            print(f"  {paired:5d} paires -> {path}")\n\n        source = candidates[0][1]\n        print(f"\\nSélection automatique: {source}\\n")\n\n    prepare(source, Path(args.out), args.size, args.overwrite)\n\n\nif __name__ == "__main__":\n    main()\n', 'train.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom pathlib import Path\n\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Subset\nfrom torch.utils.data.distributed import DistributedSampler\n\nfrom dataset import PairedPlankDataset\nfrom models import PatchDiscriminator, UNetGenerator, init_weights\nfrom utils import save_triplet_grid, seed_everything\n\n\ndef ddp_setup():\n    world_size = int(os.environ.get("WORLD_SIZE", "1"))\n    distributed = world_size > 1\n\n    if distributed:\n        local_rank = int(os.environ["LOCAL_RANK"])\n        torch.cuda.set_device(local_rank)\n        dist.init_process_group(backend="nccl")\n        rank = dist.get_rank()\n    else:\n        local_rank = 0\n        rank = 0\n\n    return distributed, rank, local_rank, world_size\n\n\ndef cleanup(distributed):\n    if distributed and dist.is_initialized():\n        dist.destroy_process_group()\n\n\ndef is_main(rank):\n    return rank == 0\n\n\ndef unwrap(model):\n    return model.module if hasattr(model, "module") else model\n\n\ndef split_indices(n, val_ratio, seed):\n    indices = list(range(n))\n    rng = random.Random(seed)\n    rng.shuffle(indices)\n\n    n_val = max(1, int(round(n * val_ratio)))\n    n_val = min(n_val, n - 1)\n\n    val_idx = indices[:n_val]\n    train_idx = indices[n_val:]\n\n    return train_idx, val_idx\n\n\n@torch.no_grad()\ndef validate(generator, loader, device, latent_channels, amp_enabled):\n    generator.eval()\n\n    l1_sum = torch.tensor(0.0, device=device)\n    n_sum = torch.tensor(0.0, device=device)\n\n    for batch in loader:\n        real = batch["image"].to(device, non_blocking=True)\n        mask = batch["mask"].to(device, non_blocking=True)\n\n        # Zéro bruit pour rendre la validation comparable epoch après epoch.\n        noise = torch.zeros(\n            real.size(0),\n            latent_channels,\n            real.size(2),\n            real.size(3),\n            device=device,\n        )\n\n        with torch.autocast(\n            device_type="cuda",\n            dtype=torch.float16,\n            enabled=amp_enabled,\n        ):\n            fake = generator(mask, noise)\n            loss = torch.mean(torch.abs(fake - real))\n\n        l1_sum += loss * real.size(0)\n        n_sum += real.size(0)\n\n    if dist.is_available() and dist.is_initialized():\n        dist.all_reduce(l1_sum, op=dist.ReduceOp.SUM)\n        dist.all_reduce(n_sum, op=dist.ReduceOp.SUM)\n\n    generator.train()\n    return (l1_sum / n_sum.clamp_min(1)).item()\n\n\ndef save_checkpoint(\n    path,\n    epoch,\n    generator,\n    discriminator,\n    opt_g,\n    opt_d,\n    scaler_g,\n    scaler_d,\n    best_val,\n    args,\n):\n    path.parent.mkdir(parents=True, exist_ok=True)\n\n    torch.save(\n        {\n            "epoch": epoch,\n            "generator": unwrap(generator).state_dict(),\n            "discriminator": unwrap(discriminator).state_dict(),\n            "optimizer_g": opt_g.state_dict(),\n            "optimizer_d": opt_d.state_dict(),\n            "scaler_g": scaler_g.state_dict(),\n            "scaler_d": scaler_d.state_dict(),\n            "best_val_l1": best_val,\n            "args": vars(args),\n        },\n        path,\n    )\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument("--data", type=str, required=True)\n    parser.add_argument("--out", type=str, required=True)\n\n    parser.add_argument("--epochs", type=int, default=150)\n    parser.add_argument("--batch", type=int, default=4, help="batch PAR GPU")\n    parser.add_argument("--workers", type=int, default=2)\n    parser.add_argument("--val-ratio", type=float, default=0.10)\n\n    parser.add_argument("--latent-channels", type=int, default=3)\n    parser.add_argument("--base", type=int, default=64)\n\n    parser.add_argument("--lr-g", type=float, default=2e-4)\n    parser.add_argument("--lr-d", type=float, default=2e-4)\n    parser.add_argument("--beta1", type=float, default=0.5)\n    parser.add_argument("--beta2", type=float, default=0.999)\n    parser.add_argument("--lambda-l1", type=float, default=50.0)\n\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--resume", type=str, default="")\n\n    args = parser.parse_args()\n\n    distributed, rank, local_rank, world_size = ddp_setup()\n\n    seed_everything(args.seed + rank)\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA n\'est pas disponible. Active un GPU Kaggle.")\n\n    device = torch.device(f"cuda:{local_rank}")\n    amp_enabled = True\n\n    out = Path(args.out)\n    ckpt_dir = out / "checkpoints"\n    sample_dir = out / "samples"\n\n    if is_main(rank):\n        ckpt_dir.mkdir(parents=True, exist_ok=True)\n        sample_dir.mkdir(parents=True, exist_ok=True)\n\n    full_train = PairedPlankDataset(args.data, augment=True)\n    full_val = PairedPlankDataset(args.data, augment=False)\n\n    train_idx, val_idx = split_indices(\n        len(full_train),\n        args.val_ratio,\n        args.seed,\n    )\n\n    train_ds = Subset(full_train, train_idx)\n    val_ds = Subset(full_val, val_idx)\n\n    train_sampler = (\n        DistributedSampler(\n            train_ds,\n            num_replicas=world_size,\n            rank=rank,\n            shuffle=True,\n            seed=args.seed,\n        )\n        if distributed\n        else None\n    )\n\n    val_sampler = (\n        DistributedSampler(\n            val_ds,\n            num_replicas=world_size,\n            rank=rank,\n            shuffle=False,\n        )\n        if distributed\n        else None\n    )\n\n    train_loader = DataLoader(\n        train_ds,\n        batch_size=args.batch,\n        shuffle=(train_sampler is None),\n        sampler=train_sampler,\n        num_workers=args.workers,\n        pin_memory=True,\n        persistent_workers=args.workers > 0,\n        drop_last=True,\n    )\n\n    val_loader = DataLoader(\n        val_ds,\n        batch_size=max(1, args.batch),\n        shuffle=False,\n        sampler=val_sampler,\n        num_workers=args.workers,\n        pin_memory=True,\n        persistent_workers=args.workers > 0,\n        drop_last=False,\n    )\n\n    G = UNetGenerator(\n        latent_channels=args.latent_channels,\n        base=args.base,\n    ).to(device)\n\n    D = PatchDiscriminator(base=args.base).to(device)\n\n    G.apply(init_weights)\n    D.apply(init_weights)\n\n    if distributed:\n        G = DDP(\n            G,\n            device_ids=[local_rank],\n            output_device=local_rank,\n            broadcast_buffers=False,\n        )\n        D = DDP(\n            D,\n            device_ids=[local_rank],\n            output_device=local_rank,\n            broadcast_buffers=False,\n        )\n\n    opt_g = torch.optim.Adam(\n        G.parameters(),\n        lr=args.lr_g,\n        betas=(args.beta1, args.beta2),\n    )\n\n    opt_d = torch.optim.Adam(\n        D.parameters(),\n        lr=args.lr_d,\n        betas=(args.beta1, args.beta2),\n    )\n\n    bce = nn.BCEWithLogitsLoss()\n    l1 = nn.L1Loss()\n\n    scaler_g = torch.amp.GradScaler("cuda", enabled=amp_enabled)\n    scaler_d = torch.amp.GradScaler("cuda", enabled=amp_enabled)\n\n    start_epoch = 1\n    best_val = float("inf")\n\n    if args.resume:\n        resume_path = Path(args.resume)\n\n        if not resume_path.exists():\n            raise FileNotFoundError(f"Checkpoint introuvable: {resume_path}")\n\n        checkpoint = torch.load(\n            resume_path,\n            map_location=device,\n            weights_only=False,\n        )\n\n        unwrap(G).load_state_dict(checkpoint["generator"])\n        unwrap(D).load_state_dict(checkpoint["discriminator"])\n\n        opt_g.load_state_dict(checkpoint["optimizer_g"])\n        opt_d.load_state_dict(checkpoint["optimizer_d"])\n\n        if "scaler_g" in checkpoint:\n            scaler_g.load_state_dict(checkpoint["scaler_g"])\n        if "scaler_d" in checkpoint:\n            scaler_d.load_state_dict(checkpoint["scaler_d"])\n\n        start_epoch = int(checkpoint["epoch"]) + 1\n        best_val = float(checkpoint.get("best_val_l1", float("inf")))\n\n        if is_main(rank):\n            print(f"Reprise depuis: {resume_path}")\n            print(f"Prochaine epoch: {start_epoch}")\n            print(f"Best val L1    : {best_val:.6f}")\n\n    if is_main(rank):\n        print("=" * 80)\n        print("GAN_PLANK EYE v2 — KAGGLE / PYTORCH")\n        print("=" * 80)\n        print(f"GPU(s)          : {world_size}")\n        print(f"Train / Val     : {len(train_ds)} / {len(val_ds)}")\n        print(f"Batch / GPU     : {args.batch}")\n        print(f"Batch global    : {args.batch * world_size}")\n        print(f"Epochs          : {args.epochs}")\n        print(f"Lambda L1       : {args.lambda_l1}")\n        print(f"AMP             : {amp_enabled}")\n        print(f"Sortie          : {out}")\n        print("=" * 80)\n\n    history = []\n\n    try:\n        for epoch in range(start_epoch, args.epochs + 1):\n            if train_sampler is not None:\n                train_sampler.set_epoch(epoch)\n\n            G.train()\n            D.train()\n\n            epoch_g = 0.0\n            epoch_d = 0.0\n            epoch_adv = 0.0\n            epoch_l1 = 0.0\n            n_batches = 0\n\n            t0 = time.time()\n\n            for step, batch in enumerate(train_loader, start=1):\n                real = batch["image"].to(device, non_blocking=True)\n                mask = batch["mask"].to(device, non_blocking=True)\n\n                noise = torch.randn(\n                    real.size(0),\n                    args.latent_channels,\n                    real.size(2),\n                    real.size(3),\n                    device=device,\n                )\n\n                # -------------------------------------------------\n                # 1. DISCRIMINATEUR\n                # -------------------------------------------------\n                opt_d.zero_grad(set_to_none=True)\n\n                with torch.autocast(\n                    device_type="cuda",\n                    dtype=torch.float16,\n                    enabled=amp_enabled,\n                ):\n                    with torch.no_grad():\n                        fake_detached = G(mask, noise)\n\n                    pred_real = D(mask, real)\n                    pred_fake = D(mask, fake_detached)\n\n                    loss_d_real = bce(\n                        pred_real,\n                        torch.ones_like(pred_real),\n                    )\n                    loss_d_fake = bce(\n                        pred_fake,\n                        torch.zeros_like(pred_fake),\n                    )\n\n                    loss_d = 0.5 * (loss_d_real + loss_d_fake)\n\n                scaler_d.scale(loss_d).backward()\n                scaler_d.step(opt_d)\n                scaler_d.update()\n\n                # -------------------------------------------------\n                # 2. GÉNÉRATEUR\n                # -------------------------------------------------\n                opt_g.zero_grad(set_to_none=True)\n\n                with torch.autocast(\n                    device_type="cuda",\n                    dtype=torch.float16,\n                    enabled=amp_enabled,\n                ):\n                    fake = G(mask, noise)\n                    pred_fake_for_g = D(mask, fake)\n\n                    loss_adv = bce(\n                        pred_fake_for_g,\n                        torch.ones_like(pred_fake_for_g),\n                    )\n\n                    loss_recon = l1(fake, real)\n                    loss_g = loss_adv + args.lambda_l1 * loss_recon\n\n                scaler_g.scale(loss_g).backward()\n                scaler_g.step(opt_g)\n                scaler_g.update()\n\n                epoch_g += loss_g.detach().item()\n                epoch_d += loss_d.detach().item()\n                epoch_adv += loss_adv.detach().item()\n                epoch_l1 += loss_recon.detach().item()\n                n_batches += 1\n\n                if is_main(rank) and (\n                    step == 1\n                    or step % 20 == 0\n                    or step == len(train_loader)\n                ):\n                    print(\n                        f"\\r[{epoch:03d}/{args.epochs}] "\n                        f"[{step:04d}/{len(train_loader):04d}] "\n                        f"G={loss_g.item():.4f} "\n                        f"D={loss_d.item():.4f} "\n                        f"ADV={loss_adv.item():.4f} "\n                        f"L1={loss_recon.item():.4f}",\n                        end="",\n                        flush=True,\n                    )\n\n            val_l1 = validate(\n                G,\n                val_loader,\n                device,\n                args.latent_channels,\n                amp_enabled,\n            )\n\n            # Moyenne train multi-GPU.\n            stats = torch.tensor(\n                [\n                    epoch_g,\n                    epoch_d,\n                    epoch_adv,\n                    epoch_l1,\n                    float(n_batches),\n                ],\n                device=device,\n            )\n\n            if distributed:\n                dist.all_reduce(stats, op=dist.ReduceOp.SUM)\n\n            denom = max(stats[4].item(), 1.0)\n            mean_g = stats[0].item() / denom\n            mean_d = stats[1].item() / denom\n            mean_adv = stats[2].item() / denom\n            mean_l1 = stats[3].item() / denom\n\n            elapsed = time.time() - t0\n\n            if is_main(rank):\n                print()\n                print(\n                    f"    train: G={mean_g:.4f} | D={mean_d:.4f} "\n                    f"| ADV={mean_adv:.4f} | L1={mean_l1:.4f} "\n                    f"| val L1={val_l1:.4f} | {elapsed/60:.1f} min"\n                )\n\n                # Preview avec le premier batch de validation.\n                preview_batch = next(iter(val_loader))\n                preview_real = preview_batch["image"].to(device)\n                preview_mask = preview_batch["mask"].to(device)\n\n                preview_noise = torch.randn(\n                    preview_real.size(0),\n                    args.latent_channels,\n                    preview_real.size(2),\n                    preview_real.size(3),\n                    device=device,\n                )\n\n                G.eval()\n                with torch.no_grad(), torch.autocast(\n                    device_type="cuda",\n                    dtype=torch.float16,\n                    enabled=amp_enabled,\n                ):\n                    preview_fake = G(preview_mask, preview_noise)\n\n                save_triplet_grid(\n                    preview_mask.cpu(),\n                    preview_fake.cpu(),\n                    preview_real.cpu(),\n                    sample_dir / f"epoch_{epoch:03d}.jpg",\n                    max_items=4,\n                )\n                G.train()\n\n                # BEST = plus petite erreur L1 validation.\n                improved = val_l1 < best_val\n                if improved:\n                    best_val = val_l1\n\n                # LAST contient toujours la meilleure valeur connue,\n                # ce qui permet une reprise propre après interruption.\n                save_checkpoint(\n                    ckpt_dir / "last.pt",\n                    epoch,\n                    G,\n                    D,\n                    opt_g,\n                    opt_d,\n                    scaler_g,\n                    scaler_d,\n                    best_val,\n                    args,\n                )\n\n                if improved:\n                    save_checkpoint(\n                        ckpt_dir / "best.pt",\n                        epoch,\n                        G,\n                        D,\n                        opt_g,\n                        opt_d,\n                        scaler_g,\n                        scaler_d,\n                        best_val,\n                        args,\n                    )\n\n                    print(\n                        f"    -> nouveau BEST: val L1 = {best_val:.6f}"\n                    )\n\n                history.append(\n                    {\n                        "epoch": epoch,\n                        "train_g": mean_g,\n                        "train_d": mean_d,\n                        "train_adv": mean_adv,\n                        "train_l1": mean_l1,\n                        "val_l1": val_l1,\n                        "seconds": elapsed,\n                    }\n                )\n\n                (out / "history.json").write_text(\n                    json.dumps(history, indent=2),\n                    encoding="utf-8",\n                )\n\n    except KeyboardInterrupt:\n        if is_main(rank):\n            print(\n                "\\nInterruption manuelle détectée. "\n                "best.pt et last.pt des epochs déjà terminées restent sauvegardés."\n            )\n        raise\n    finally:\n        cleanup(distributed)\n\n\nif __name__ == "__main__":\n    main()\n', 'generate.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport random\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nfrom models import UNetGenerator\n\n\ndef polygon_mask(points, size):\n    mask = np.zeros((size, size), dtype=np.uint8)\n    cv2.fillPoly(mask, [points.astype(np.int32)], 255)\n    return mask\n\n\ndef order_clockwise(points):\n    points = np.asarray(points, dtype=np.float32)\n    center = points.mean(axis=0)\n\n    angles = np.arctan2(\n        points[:, 1] - center[1],\n        points[:, 0] - center[0],\n    )\n\n    ordered = points[np.argsort(angles)]\n\n    # Commence par le coin le plus proche du haut-gauche,\n    # puis conserve l\'ordre circulaire: TL-ish, TR-ish, BR-ish, BL-ish.\n    start = int(np.argmin(ordered[:, 0] + ordered[:, 1]))\n    ordered = np.roll(ordered, -start, axis=0)\n\n    return ordered\n\n\ndef rotated_plank(size, rng):\n    """\n    Génère une planche sous forme de rectangle orienté,\n    légèrement perturbé pour obtenir un quadrilatère réaliste.\n    """\n    w = rng.uniform(0.22, 0.55) * size\n    h = rng.uniform(0.07, 0.18) * size\n\n    # Certaines images contiennent des planches plutôt verticales.\n    angle = rng.uniform(-80.0, 80.0)\n\n    margin = max(w, h) * 0.65 + 4\n    cx = rng.uniform(margin, size - margin)\n    cy = rng.uniform(margin, size - margin)\n\n    rect = ((cx, cy), (w, h), angle)\n    pts = cv2.boxPoints(rect).astype(np.float32)\n\n    # Petit bruit sur chaque coin.\n    jitter = min(w, h) * 0.04\n    pts += np.asarray(\n        [\n            [rng.uniform(-jitter, jitter), rng.uniform(-jitter, jitter)]\n            for _ in range(4)\n        ],\n        dtype=np.float32,\n    )\n\n    pts[:, 0] = np.clip(pts[:, 0], 2, size - 3)\n    pts[:, 1] = np.clip(pts[:, 1], 2, size - 3)\n\n    return order_clockwise(pts)\n\n\ndef overlap_ratio(mask_a, mask_b):\n    inter = np.logical_and(mask_a > 0, mask_b > 0).sum()\n    area_b = (mask_b > 0).sum()\n\n    if area_b == 0:\n        return 1.0\n\n    return float(inter) / float(area_b)\n\n\ndef too_close(existing_mask, new_mask, min_gap):\n    if min_gap <= 0:\n        return False\n\n    k = 2 * int(min_gap) + 1\n    kernel = np.ones((k, k), dtype=np.uint8)\n\n    dilated = cv2.dilate(\n        existing_mask,\n        kernel,\n        iterations=1,\n    )\n\n    return bool(\n        np.logical_and(dilated > 0, new_mask > 0).any()\n    )\n\n\ndef build_layout(\n    size,\n    plank_count,\n    min_gap,\n    max_overlap,\n    rng,\n    max_attempts_per_plank=250,\n):\n    occupancy = np.zeros((size, size), dtype=np.uint8)\n    polygons = []\n\n    for _ in range(plank_count):\n        accepted = False\n\n        for _attempt in range(max_attempts_per_plank):\n            pts = rotated_plank(size, rng)\n            pmask = polygon_mask(pts, size)\n\n            overlap = overlap_ratio(occupancy, pmask)\n\n            if overlap > max_overlap:\n                continue\n\n            # Avec max_overlap=0, on applique aussi l\'écart minimal.\n            if max_overlap <= 0.0 and too_close(\n                occupancy,\n                pmask,\n                min_gap,\n            ):\n                continue\n\n            occupancy = np.maximum(occupancy, pmask)\n            polygons.append(pts)\n            accepted = True\n            break\n\n        if not accepted:\n            return None, None\n\n    return occupancy, polygons\n\n\ndef save_label(path, polygons, size):\n    lines = []\n\n    for pts in polygons:\n        norm = pts.astype(np.float32).copy()\n        norm[:, 0] /= float(size)\n        norm[:, 1] /= float(size)\n        norm = np.clip(norm, 0.0, 1.0)\n\n        coords = " ".join(\n            f"{v:.6f}" for v in norm.reshape(-1)\n        )\n        lines.append(f"0 {coords}")\n\n    path.write_text(\n        "\\n".join(lines) + "\\n",\n        encoding="utf-8",\n    )\n\n\ndef load_generator(checkpoint_path, device, latent_channels, base):\n    checkpoint = torch.load(\n        checkpoint_path,\n        map_location=device,\n        weights_only=False,\n    )\n\n    ckpt_args = checkpoint.get("args", {})\n\n    latent_channels = int(\n        ckpt_args.get("latent_channels", latent_channels)\n    )\n    base = int(\n        ckpt_args.get("base", base)\n    )\n\n    model = UNetGenerator(\n        latent_channels=latent_channels,\n        base=base,\n    ).to(device)\n\n    model.load_state_dict(checkpoint["generator"])\n    model.eval()\n\n    return model, latent_channels\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument("--checkpoint", type=str, required=True)\n    parser.add_argument("--out", type=str, required=True)\n\n    # Gardé pour compatibilité avec notre ancienne commande.\n    parser.add_argument("--prepared", type=str, default="")\n\n    parser.add_argument("--count", type=int, default=1000)\n    parser.add_argument("--size", type=int, default=512)\n\n    parser.add_argument("--min-planks", type=int, default=1)\n    parser.add_argument("--max-planks", type=int, default=7)\n    parser.add_argument("--min-gap", type=int, default=8)\n    parser.add_argument("--max-overlap", type=float, default=0.0)\n\n    parser.add_argument("--latent-channels", type=int, default=3)\n    parser.add_argument("--base", type=int, default=64)\n    parser.add_argument("--seed", type=int, default=12345)\n\n    args = parser.parse_args()\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA non disponible.")\n\n    device = torch.device("cuda")\n\n    out = Path(args.out)\n    image_dir = out / "images"\n    label_dir = out / "labels"\n    mask_dir = out / "masks"\n\n    image_dir.mkdir(parents=True, exist_ok=True)\n    label_dir.mkdir(parents=True, exist_ok=True)\n    mask_dir.mkdir(parents=True, exist_ok=True)\n\n    G, latent_channels = load_generator(\n        args.checkpoint,\n        device,\n        args.latent_channels,\n        args.base,\n    )\n\n    rng = random.Random(args.seed)\n    torch_gen = torch.Generator(device=device)\n    torch_gen.manual_seed(args.seed)\n\n    produced = 0\n    failed_layouts = 0\n\n    metadata = []\n\n    while produced < args.count:\n        plank_count = rng.randint(\n            args.min_planks,\n            args.max_planks,\n        )\n\n        occupancy, polygons = build_layout(\n            size=args.size,\n            plank_count=plank_count,\n            min_gap=args.min_gap,\n            max_overlap=args.max_overlap,\n            rng=rng,\n        )\n\n        if occupancy is None:\n            failed_layouts += 1\n\n            if failed_layouts > args.count * 50:\n                raise RuntimeError(\n                    "Impossible de construire suffisamment de layouts. "\n                    "Réduis min-gap / max-planks ou augmente max-overlap."\n                )\n            continue\n\n        mask = torch.from_numpy(\n            (occupancy.astype(np.float32) / 255.0)[None, None]\n        ).to(device)\n\n        noise = torch.randn(\n            1,\n            latent_channels,\n            args.size,\n            args.size,\n            generator=torch_gen,\n            device=device,\n        )\n\n        with torch.no_grad(), torch.autocast(\n            device_type="cuda",\n            dtype=torch.float16,\n            enabled=True,\n        ):\n            fake = G(mask, noise)\n\n        image = (\n            (fake[0].float().clamp(-1, 1) + 1.0)\n            * 127.5\n        )\n        image = (\n            image.permute(1, 2, 0)\n            .cpu()\n            .numpy()\n            .round()\n            .clip(0, 255)\n            .astype(np.uint8)\n        )\n\n        stem = f"gan_{produced:06d}"\n\n        Image.fromarray(image).save(\n            image_dir / f"{stem}.jpg",\n            quality=95,\n        )\n\n        cv2.imwrite(\n            str(mask_dir / f"{stem}.png"),\n            occupancy,\n        )\n\n        save_label(\n            label_dir / f"{stem}.txt",\n            polygons,\n            args.size,\n        )\n\n        metadata.append(\n            {\n                "stem": stem,\n                "plank_count": plank_count,\n            }\n        )\n\n        produced += 1\n\n        if produced == 1 or produced % 50 == 0 or produced == args.count:\n            print(\n                f"\\rGénération: {produced}/{args.count}",\n                end="",\n                flush=True,\n            )\n\n    print()\n\n    (out / "metadata.json").write_text(\n        json.dumps(\n            {\n                "checkpoint": args.checkpoint,\n                "count": produced,\n                "size": args.size,\n                "min_planks": args.min_planks,\n                "max_planks": args.max_planks,\n                "min_gap": args.min_gap,\n                "max_overlap": args.max_overlap,\n                "seed": args.seed,\n                "items": metadata,\n            },\n            indent=2,\n        ),\n        encoding="utf-8",\n    )\n\n    print("=" * 72)\n    print("GÉNÉRATION TERMINÉE")\n    print("=" * 72)\n    print(f"Images : {image_dir}")\n    print(f"Labels : {label_dir}")\n    print(f"Masks  : {mask_dir}")\n    print(f"Total  : {produced}")\n\n\nif __name__ == "__main__":\n    main()\n'}

for name, content in FILES.items():
    path = PROJECT / name
    path.write_text(content, encoding="utf-8")
    print("créé:", path)

print("\nProjet:", PROJECT)


créé: /kaggle/working/GAN_PlankEye_v2/config.py
créé: /kaggle/working/GAN_PlankEye_v2/utils.py
créé: /kaggle/working/GAN_PlankEye_v2/dataset.py
créé: /kaggle/working/GAN_PlankEye_v2/models.py
créé: /kaggle/working/GAN_PlankEye_v2/prepare_dataset.py
créé: /kaggle/working/GAN_PlankEye_v2/train.py
créé: /kaggle/working/GAN_PlankEye_v2/generate.py

Projet: /kaggle/working/GAN_PlankEye_v2


In [11]:
from pathlib import Path

INPUT = Path("/kaggle/input")

print("=== CONTENU DE /kaggle/input ===")

for path in sorted(INPUT.rglob("*")):
    if path.is_dir():
        print(path)

=== CONTENU DE /kaggle/input ===
/kaggle/input/datasets
/kaggle/input/datasets/max778
/kaggle/input/datasets/max778/gan-plankeye-checkpoints
/kaggle/input/datasets/max778/plankeye-gan
/kaggle/input/datasets/max778/plankeye-gan/data32
/kaggle/input/datasets/max778/plankeye-gan/data32/images
/kaggle/input/datasets/max778/plankeye-gan/data32/labels
/kaggle/input/datasets/max778/plankeye-gan/data32/viz_gt


In [12]:
%cd /kaggle/working/GAN_PlankEye_v2

import subprocess

SOURCE = "/kaggle/input/datasets/max778/plankeye-gan/data_gan"

cmd = [
    "python",
    "prepare_dataset.py",
    "--source", SOURCE,
    "--out", "/kaggle/working/GAN_PlankEye_v2/data/paired",
    "--size", "512",
    "--overwrite",
]

print("Commande :")
print(" ".join(cmd))

subprocess.run(cmd, check=True)

/kaggle/working/GAN_PlankEye_v2
Commande :
python prepare_dataset.py --source /kaggle/input/datasets/max778/plankeye-gan/data32 --out /kaggle/working/GAN_PlankEye_v2/data/paired --size 512 --overwrite
PRÉPARATION TERMINÉE
Source                : /kaggle/input/datasets/max778/plankeye-gan/data32
Sortie                : /kaggle/working/GAN_PlankEye_v2/data/paired
Taille                : 512x512
Paires conservées     : 294
Sans label            : 0
Labels vides/invalides: 0
Distribution nb planches:
  1: 100
  2: 70
  3: 100
  4: 11
  6: 10
  7: 3


CompletedProcess(args=['python', 'prepare_dataset.py', '--source', '/kaggle/input/datasets/max778/plankeye-gan/data32', '--out', '/kaggle/working/GAN_PlankEye_v2/data/paired', '--size', '512', '--overwrite'], returncode=0)

## Préparer le dataset

Le script cherche automatiquement dans `/kaggle/input` le plus gros dossier contenant à la fois `images/` et `labels/`.

Si Kaggle choisit le mauvais dataset, renseigne `SOURCE` manuellement.

In [13]:
%cd /kaggle/working/GAN_PlankEye_v2

# Laisse vide pour détection automatique.
SOURCE = "/kaggle/input/plankeye-gan/data_gan"

cmd = [
    "python", "prepare_dataset.py",
    "--out", "/kaggle/working/GAN_PlankEye_v2/data/paired",
    "--size", "512",
    "--overwrite",
]

if SOURCE:
    cmd += ["--source", SOURCE]

import subprocess
subprocess.run(cmd, check=True)

/kaggle/working/GAN_PlankEye_v2


Traceback (most recent call last):
  File "/kaggle/working/GAN_PlankEye_v2/prepare_dataset.py", line 188, in <module>
    main()
  File "/kaggle/working/GAN_PlankEye_v2/prepare_dataset.py", line 184, in main
    prepare(source, Path(args.out), args.size, args.overwrite)
  File "/kaggle/working/GAN_PlankEye_v2/prepare_dataset.py", line 55, in prepare
    raise FileNotFoundError(
FileNotFoundError: Le dossier source doit contenir `images/` et `labels/`.
Source reçue: /kaggle/input/plankeye-gan/data32


CalledProcessError: Command '['python', 'prepare_dataset.py', '--out', '/kaggle/working/GAN_PlankEye_v2/data/paired', '--size', '512', '--overwrite', '--source', '/kaggle/input/plankeye-gan/data32']' returned non-zero exit status 1.

In [ ]:
from pathlib import Path
import json

paired = Path("/kaggle/working/GAN_PlankEye_v2/data/paired")
meta = json.loads((paired / "metadata.json").read_text())

print("Paires :", meta["pairs"])
print("Distribution :", meta["plank_count_distribution"])
print("Images :", len(list((paired / "images").glob("*"))))
print("Masks  :", len(list((paired / "masks").glob("*.png"))))
print("Labels :", len(list((paired / "labels").glob("*.txt"))))

## Visualiser quelques paires réelles / masques

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import random

paired = Path("/kaggle/working/GAN_PlankEye_v2/data/paired")
images = list((paired / "images").glob("*.jpg"))
random.shuffle(images)

for p in images[:3]:
    mask = paired / "masks" / f"{p.stem}.png"

    fig = plt.figure(figsize=(10, 4))
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.imshow(Image.open(p))
    ax1.set_title(p.name)
    ax1.axis("off")

    ax2 = fig.add_subplot(1, 2, 2)
    ax2.imshow(Image.open(mask), cmap="gray")
    ax2.set_title("Condition GAN")
    ax2.axis("off")

    plt.show()

## Entraîner

Pour Kaggle avec **2×T4**, garde `NPROC=2`.
Avec un seul GPU, mets `NPROC=1`.

In [ ]:
%cd /kaggle/working/GAN_PlankEye_v2

NPROC = min(2, torch.cuda.device_count())

train_cmd = [
    "torchrun",
    "--standalone",
    f"--nproc_per_node={NPROC}",
    "train.py",
    "--data", "/kaggle/working/GAN_PlankEye_v2/data/paired",
    "--out", "/kaggle/working/GAN_PlankEye_v2/runs/plankgan_multi_512",
    "--epochs", "150",
    "--batch", "4",
    "--workers", "2",
    "--lambda-l1", "50",
]

print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True)

## Reprendre depuis `last.pt`

Exécute cette cellule **à la place de la cellule d'entraînement précédente** pour continuer un entraînement interrompu.

In [ ]:
%cd /kaggle/working/GAN_PlankEye_v2

LAST = Path(
    "/kaggle/working/GAN_PlankEye_v2/"
    "runs/plankgan_multi_512/checkpoints/last.pt"
)

assert LAST.exists(), LAST

NPROC = min(2, torch.cuda.device_count())

resume_cmd = [
    "torchrun",
    "--standalone",
    f"--nproc_per_node={NPROC}",
    "train.py",
    "--data", "/kaggle/working/GAN_PlankEye_v2/data/paired",
    "--out", "/kaggle/working/GAN_PlankEye_v2/runs/plankgan_multi_512",
    "--epochs", "150",
    "--batch", "4",
    "--workers", "2",
    "--lambda-l1", "50",
    "--resume", str(LAST),
]

print(" ".join(resume_cmd))
subprocess.run(resume_cmd, check=True)

## Afficher la dernière preview

In [ ]:
from IPython.display import display
from PIL import Image

sample_dir = Path(
    "/kaggle/working/GAN_PlankEye_v2/"
    "runs/plankgan_multi_512/samples"
)

samples = sorted(sample_dir.glob("epoch_*.jpg"))

if samples:
    print(samples[-1])
    display(Image.open(samples[-1]))
else:
    print("Aucune preview pour le moment.")

## Générer 1000 images synthétiques avec labels

In [ ]:
%cd /kaggle/working/GAN_PlankEye_v2

BEST = Path(
    "/kaggle/working/GAN_PlankEye_v2/"
    "runs/plankgan_multi_512/checkpoints/best.pt"
)

assert BEST.exists(), BEST

generate_cmd = [
    "python", "generate.py",
    "--checkpoint", str(BEST),
    "--prepared", "/kaggle/working/GAN_PlankEye_v2/data/paired",
    "--out", "/kaggle/working/GAN_PlankEye_v2/generated_v2_512",
    "--count", "1000",
    "--size", "512",
    "--min-planks", "1",
    "--max-planks", "7",
    "--min-gap", "8",
    "--max-overlap", "0",
]

print(" ".join(generate_cmd))
subprocess.run(generate_cmd, check=True)

## Vérifier les images générées + polygones

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

gen_root = Path("/kaggle/working/GAN_PlankEye_v2/generated_v2_512")
gen_images = sorted((gen_root / "images").glob("*.jpg"))

for p in gen_images[:5]:
    img = cv2.imread(str(p))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    label_path = gen_root / "labels" / f"{p.stem}.txt"

    for line in label_path.read_text().splitlines():
        parts = line.split()
        coords = np.asarray(list(map(float, parts[1:9]))).reshape(4, 2)
        coords[:, 0] *= img.shape[1]
        coords[:, 1] *= img.shape[0]
        pts = coords.astype(np.int32)
        cv2.polylines(img, [pts], True, (255, 255, 255), 2)

    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(p.name)
    plt.axis("off")
    plt.show()

## ZIP final

Crée une archive avec les images et labels générés pour pouvoir la sauvegarder comme Output Kaggle ou créer un Dataset Kaggle.

In [ ]:
import shutil
from pathlib import Path

src = Path("/kaggle/working/GAN_PlankEye_v2/generated_v2_512")
zip_base = "/kaggle/working/GAN_PlankEye_v2/generated_v2_512"

archive = shutil.make_archive(zip_base, "zip", src)
print("Archive:", archive)